In [0]:
%pip install torch 

In [0]:
import random
from copy import deepcopy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import GroupShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_recall_curve,
    confusion_matrix
)

SEED = 42
BATCH_SIZE = 256
LEARNING_RATE = 0.001
MAX_EPOCHS = 50
PATIENCE = 7
ALERT_FRACTION = 0.05

# Enable only after confirming this count was known before the transaction.
USE_PRIOR_FRAUD = False

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("PyTorch:", torch.__version__)

In [0]:
transactions = (
    spark.table("fraud_transactions_for_detection")
    .toPandas()
    .reset_index(drop=True)
)

assert transactions["Transaction_ID"].notna().all()
assert transactions["Transaction_ID"].is_unique
assert transactions["User_ID"].notna().all()
assert transactions["Fraudulent"].notna().all()
assert transactions["Fraudulent"].isin([0, 1]).all()

first_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=SEED
)

development_idx, test_idx = next(
    first_split.split(transactions, groups=transactions["User_ID"])
)

development = transactions.iloc[development_idx].copy()
test = transactions.iloc[test_idx].copy()

second_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=SEED
)

train_idx, val_idx = next(
    second_split.split(development, groups=development["User_ID"])
)

train = development.iloc[train_idx].copy()
validation = development.iloc[val_idx].copy()

assert set(train["User_ID"]).isdisjoint(validation["User_ID"])
assert set(train["User_ID"]).isdisjoint(test["User_ID"])
assert set(validation["User_ID"]).isdisjoint(test["User_ID"])
assert len(train) + len(validation) + len(test) == len(transactions)

display(pd.DataFrame([
    {
        "set": name,
        "rows": len(frame),
        "users": frame["User_ID"].nunique(),
        "fraud_cases": int(frame["Fraudulent"].sum()),
        "fraud_rate_pct": 100 * frame["Fraudulent"].mean()
    }
    for name, frame in [
        ("Training", train),
        ("Validation", validation),
        ("Test", test)
    ]
]))

In [0]:
categorical_features = [
    "Transaction_Type",
    "Device_Used",
    "Location",
    "Payment_Method"
]

numeric_features = [
    "Transaction_Amount",
    "Account_Age",
    "Number_of_Transactions_Last_24H",
    "hour_sin",
    "hour_cos"
]

if USE_PRIOR_FRAUD:
    numeric_features.append("Previous_Fraudulent_Transactions")


def make_features(frame):
    features = frame.copy()

    hour = pd.to_numeric(
        features["Time_of_Transaction"],
        errors="raise"
    )

    assert hour.dropna().between(0, 23).all()

    features["hour_sin"] = np.sin(2 * np.pi * hour / 24)
    features["hour_cos"] = np.cos(2 * np.pi * hour / 24)

    for column in numeric_features:
        features[column] = pd.to_numeric(
            features[column],
            errors="raise"
        )
        assert not np.isinf(features[column].to_numpy(dtype=float)).any()

    for column in categorical_features:
        features[column] = (
            features[column]
            .astype("string")
            .fillna("__MISSING__")
            .astype(str)
        )

    # Explicit selection prevents IDs and labels entering the model.
    return features[numeric_features + categorical_features]


train_features = make_features(train)
val_features = make_features(validation)

assert train_features[numeric_features].notna().any().all()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    (
        "categorical",
        OneHotEncoder(handle_unknown="ignore", sparse_output=False),
        categorical_features
    )
])

X_train = preprocessor.fit_transform(train_features).astype(np.float32)
X_val = preprocessor.transform(val_features).astype(np.float32)

y_train = train["Fraudulent"].to_numpy(dtype=np.float32)
y_val = validation["Fraudulent"].to_numpy(dtype=np.float32)

assert np.isfinite(X_train).all()
assert np.isfinite(X_val).all()
assert X_train.shape[1] == X_val.shape[1]
assert set(np.unique(y_train)) == {0, 1}
assert set(np.unique(y_val)) == {0, 1}

print("Training inputs:", X_train.shape)
print("Validation inputs:", X_val.shape)

In [0]:
class FraudMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.10),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.10),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x).squeeze(-1)


torch.manual_seed(SEED)

model = FraudMLP(input_dim=X_train.shape[1])

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=0.0001
)

training_dataset = TensorDataset(
    torch.from_numpy(X_train),
    torch.from_numpy(y_train)
)

train_loader = DataLoader(
    training_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED)
)

with torch.no_grad():
    example_output = model(torch.from_numpy(X_train[:8]))

assert example_output.shape == (8,)
print(model)

In [0]:
def predict_scores(inputs):
    model.eval()
    predictions = []

    with torch.no_grad():
        for start in range(0, len(inputs), BATCH_SIZE):
            batch = torch.from_numpy(inputs[start:start + BATCH_SIZE])
            scores = torch.sigmoid(model(batch))
            predictions.append(scores.cpu().numpy())

    return np.concatenate(predictions)


best_ap = -np.inf
best_weights = None
best_epoch = None
stale_epochs = 0
history = []

validation_tensor = torch.from_numpy(X_val)
validation_targets = torch.from_numpy(y_val)

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    total_loss = 0.0

    for inputs, targets in train_loader:
        optimizer.zero_grad(set_to_none=True)

        logits = model(inputs)
        loss = criterion(logits, targets)

        if not torch.isfinite(loss).item():
            raise RuntimeError("Training loss became non-finite.")

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=5.0,
            error_if_nonfinite=True
        )

        optimizer.step()
        total_loss += loss.item() * len(inputs)

    model.eval()

    with torch.no_grad():
        val_logits = model(validation_tensor)
        val_loss = criterion(val_logits, validation_targets).item()
        val_scores_epoch = torch.sigmoid(val_logits).cpu().numpy()

    assert np.isfinite(val_scores_epoch).all()

    val_ap = average_precision_score(y_val, val_scores_epoch)
    train_loss = total_loss / len(training_dataset)

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_ap": val_ap
    })

    print(
        f"Epoch {epoch:02d} | "
        f"Train loss: {train_loss:.4f} | "
        f"Val loss: {val_loss:.4f} | "
        f"Val AP: {val_ap:.4f}"
    )

    if val_ap > best_ap + 0.0001:
        best_ap = val_ap
        best_weights = deepcopy(model.state_dict())
        best_epoch = epoch
        stale_epochs = 0
    else:
        stale_epochs += 1

    if stale_epochs >= PATIENCE:
        print("Early stopping.")
        break

model.load_state_dict(best_weights)
model.eval()

history_df = pd.DataFrame(history)

print(f"\nBest epoch: {best_epoch}")
print(f"Best validation AP: {best_ap:.4f}")
print(f"Validation fraud prevalence: {y_val.mean():.4f}")

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(history_df["epoch"], history_df["train_loss"], label="Training")
axes[0].plot(history_df["epoch"], history_df["val_loss"], label="Validation")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Binary cross-entropy")
axes[0].legend()

axes[1].plot(history_df["epoch"], history_df["val_ap"])
axes[1].axhline(
    y_val.mean(),
    linestyle="--",
    color="grey",
    label="Fraud prevalence"
)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Validation average precision")
axes[1].legend()

plt.tight_layout()
plt.show()

val_scores = predict_scores(X_val)

# Set an initial threshold for approximately 5% validation alerts.
threshold = float(np.quantile(val_scores, 1 - ALERT_FRACTION))


def evaluate_predictions(labels, scores, cutoff, name):
    predictions = (scores >= cutoff).astype(int)
    tn, fp, fn, tp = confusion_matrix(
        labels, predictions, labels=[0, 1]
    ).ravel()

    precision = tp / (tp + fp) if tp + fp else np.nan
    recall = tp / (tp + fn) if tp + fn else np.nan
    prevalence = labels.mean()

    return {
        "set": name,
        "threshold": cutoff,
        "transactions": len(labels),
        "alerts": int(tp + fp),
        "alert_rate_pct": 100 * predictions.mean(),
        "true_positives": int(tp),
        "false_positives": int(fp),
        "false_negatives": int(fn),
        "true_negatives": int(tn),
        "precision_pct": 100 * precision,
        "recall_pct": 100 * recall,
        "lift": precision / prevalence if prevalence > 0 else np.nan,
        "average_precision": average_precision_score(labels, scores),
        "roc_auc": roc_auc_score(labels, scores)
    }


validation_metrics = evaluate_predictions(
    y_val, val_scores, threshold, "Validation"
)

display(pd.DataFrame([validation_metrics]))

precision_curve, recall_curve, _ = precision_recall_curve(
    y_val, val_scores
)

plt.figure(figsize=(6, 4))
plt.plot(recall_curve, precision_curve, label="MLP")
plt.axhline(
    y_val.mean(),
    linestyle="--",
    color="grey",
    label="Fraud prevalence"
)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Validation precision–recall curve")
plt.legend()
plt.tight_layout()
plt.show()

In [0]:
final_threshold = threshold

test_features = make_features(test)
X_test = preprocessor.transform(test_features).astype(np.float32)
y_test = test["Fraudulent"].to_numpy(dtype=np.float32)

assert np.isfinite(X_test).all()
assert set(np.unique(y_test)) == {0, 1}

test_scores = predict_scores(X_test)

test_metrics = evaluate_predictions(
    y_test,
    test_scores,
    final_threshold,
    "Test"
)

display(pd.DataFrame([
    validation_metrics,
    test_metrics
]))

test_results = test[
    ["Transaction_ID", "User_ID", "Fraudulent"]
].copy()

test_results["fraud_score"] = test_scores.astype(float)
test_results["predicted_fraud"] = (
    test_scores >= final_threshold
).astype(int)

display(
    test_results.sort_values(
        "fraud_score",
        ascending=False
    ).head(20)
)

In [0]:
spark.createDataFrame(test_results).createOrReplaceTempView(
    "mlp_test_predictions"
)



In [0]:
%sql
SELECT
    predicted_fraud,
    COUNT(*) AS transaction_count,
    SUM(Fraudulent) AS actual_fraud_cases,
    ROUND(100.0 * AVG(Fraudulent), 2) AS actual_fraud_rate_pct,
    ROUND(AVG(fraud_score), 4) AS average_model_score
FROM mlp_test_predictions
GROUP BY predicted_fraud
ORDER BY predicted_fraud DESC;

In [0]:
%sql
SELECT
    a.predicted_fraud AS rule_prediction,
    COUNT(*) AS transaction_count,
    SUM(p.Fraudulent) AS actual_fraud_cases,
    ROUND(100.0 * AVG(p.Fraudulent), 2) AS actual_fraud_rate_pct
FROM mlp_test_predictions AS p
INNER JOIN fraud_alerts AS a
    ON p.Transaction_ID = a.Transaction_ID
GROUP BY a.predicted_fraud
ORDER BY rule_prediction DESC;

In [0]:
threshold_results = []

for alert_fraction in [0.01, 0.05, 0.10, 0.20, 0.30, 0.50]:
    candidate_threshold = float(
        np.quantile(val_scores, 1 - alert_fraction)
    )

    metrics = evaluate_predictions(
        y_val,
        val_scores,
        candidate_threshold,
        "Validation"
    )

    metrics["target_alert_rate_pct"] = 100 * alert_fraction
    threshold_results.append(metrics)

display(
    pd.DataFrame(threshold_results)[[
        "target_alert_rate_pct",
        "threshold",
        "alert_rate_pct",
        "true_positives",
        "false_positives",
        "precision_pct",
        "recall_pct",
        "lift"
    ]]
)